<a href="https://colab.research.google.com/github/siddhartha-sai-17/Transformer/blob/main/Transfomer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Attention Mechnasim***

In [38]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    LSTM,
    Attention,
    Embedding,
    Concatenate,
    Softmax,
    Dot
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [3]:
english_sentences = [
    "i love ai",
    "i love deep learning",
    "how are you",
    "good morning"
]

french_sentences = [
    "start j aime ai end",
    "start j aime apprentissage profond end",
    "start comment allez vous end",
    "start bonjour end"
]


**Tokenization**

In [5]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
fra_tokenizer = Tokenizer()
fra_tokenizer.fit_on_texts(french_sentences)
print("English Vocabulary Size:", len(eng_tokenizer.word_index) + 1)
print("French Vocabulary Size:", len(fra_tokenizer.word_index) + 1)
print("English Vocabulary: \n", eng_tokenizer.word_index)
print("French Vocabulary: \n", fra_tokenizer.word_index)

English Vocabulary Size: 11
French Vocabulary Size: 12
English Vocabulary: 
 {'i': 1, 'love': 2, 'ai': 3, 'deep': 4, 'learning': 5, 'how': 6, 'are': 7, 'you': 8, 'good': 9, 'morning': 10}
French Vocabulary: 
 {'start': 1, 'end': 2, 'j': 3, 'aime': 4, 'ai': 5, 'apprentissage': 6, 'profond': 7, 'comment': 8, 'allez': 9, 'vous': 10, 'bonjour': 11}


**Convert Sentence into Sequence**

In [25]:
encoder_sequences = eng_tokenizer.texts_to_sequences(english_sentences)
decoder_sequences = fra_tokenizer.texts_to_sequences(french_sentences)
print("Encoder Sequences: \n", encoder_sequences)
print("Decoder Sequences: \n", decoder_sequences)

Encoder Sequences: 
 [[1, 2, 3], [1, 2, 4, 5], [6, 7, 8], [9, 10]]
Decoder Sequences: 
 [[1, 3, 4, 5, 2], [1, 3, 4, 6, 7, 2], [1, 8, 9, 10, 2], [1, 11, 2]]


**Padding**

Padding makes the all sequence equal length

In [26]:
encoder_input_data = pad_sequences(encoder_sequences, padding="post")
decoder_input_data = pad_sequences(decoder_sequences, padding="post")
print("Encoder Input Data for Training: \n", encoder_input_data)
print("Decoder Input Data for Training: \n", decoder_input_data)

Encoder Input Data for Training: 
 [[ 1  2  3  0]
 [ 1  2  4  5]
 [ 6  7  8  0]
 [ 9 10  0  0]]
Decoder Input Data for Training: 
 [[ 1  3  4  5  2  0]
 [ 1  3  4  6  7  2]
 [ 1  8  9 10  2  0]
 [ 1 11  2  0  0  0]]


**Build Encoder**

Encoder generates hidden states for every word

In [39]:
tf.keras.backend.clear_session()

encoder_input_tensor = Input(shape = (None, ))
encoder_embedding = Embedding(
    input_dim = len(eng_tokenizer.word_index) + 1,
    output_dim = 64,
)(encoder_input_tensor)

encoder_outputs , state_h, state_c = LSTM(
    units = 64,
    return_state = True,
    return_sequences = True,
)(encoder_embedding)
print("Encoder Created successfully")

Encoder Created successfully


**Import Understanding**

variable ---> meaning

encoder_outputs--->  hidden states for all words

state_h ---> final hidden state

state_c ---> final cell state

**Build decoder**

In [33]:
decoder_input_tensor = Input(shape=(None,))
decoder_embedding = Embedding(
    input_dim = len(fra_tokenizer.word_index) + 1,
    output_dim = 64,
)(decoder_input_tensor)

decoder_sequences_output, decoder_state_h, decoder_state_c = LSTM(
    units = 64,
    return_state = True,
    return_sequences = True,
)(decoder_embedding, initial_state = [state_h, state_c])
print("Decoder Created successfully")

Decoder Created successfully


**Attention Layer**

Attention compares:


*   decodes output with
*   encoder outputs

and dynamically focuses on important words

In [41]:
decoder_combined = decoder_sequences_output

**Combine Decoder + Attention Output**

In [35]:
decoder_combined = Concatenate()([decoder_sequences_output, attention_output])
print("Decoder + Attention Combined successfully")

Decoder + Attention Combined successfully


**Final Prediction Layer**

In [43]:
output = Dense(
    len(fra_tokenizer.word_index)+1,
    activation='softmax'
)(decoder_combined)
print("Output Layer Created successfully")

Output Layer Created successfully


**Build Model**

In [24]:
model = Model(
    [encoder_input_tensor, decoder_input_tensor],
    output
)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │        704 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │        768 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │     33,024 │ embedding[0][0]   │
│                     │ 64), (None, 64),  │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     33,024 │ embedding_1[0][0… │
│                     │ 64), (None, 64),  │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 64)  │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 128) │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 12)  │      1,548 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 69,068 (269.80 KB)

 Trainable params: 69,068 (269.80 KB)

 Non-trainable params: 0 (0.00 B)

**Compile Model**


In [31]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)
print("Model Compiled successfully")

Model Compiled successfully


**Train Model**

In [45]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =====================================================
# Dataset
# =====================================================

english_sentences = [
    "i love ai",
    "i love deep learning",
    "how are you",
    "good morning"
]

french_sentences = [
    "start j aime ai end",
    "start j aime apprentissage profond end",
    "start comment allez vous end",
    "start bonjour end"
]

# =====================================================
# Tokenization
# =====================================================

eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

fra_tokenizer = Tokenizer()
fra_tokenizer.fit_on_texts(french_sentences)

eng_vocab_size = len(eng_tokenizer.word_index) + 1
fra_vocab_size = len(fra_tokenizer.word_index) + 1

print("English Vocabulary Size:", eng_vocab_size)
print("French Vocabulary Size:", fra_vocab_size)

print("\nEnglish Vocabulary:")
print(eng_tokenizer.word_index)

print("\nFrench Vocabulary:")
print(fra_tokenizer.word_index)

# =====================================================
# Convert Text to Sequences
# =====================================================

encoder_sequences = eng_tokenizer.texts_to_sequences(
    english_sentences
)

decoder_sequences = fra_tokenizer.texts_to_sequences(
    french_sentences
)

print("\nEncoder Sequences:")
print(encoder_sequences)

print("\nDecoder Sequences:")
print(decoder_sequences)

# =====================================================
# Padding
# =====================================================

encoder_input_data = pad_sequences(
    encoder_sequences,
    padding="post"
)

decoder_input_data = pad_sequences(
    decoder_sequences,
    padding="post"
)

print("\nEncoder Input Data:")
print(encoder_input_data)

print("\nDecoder Input Data:")
print(decoder_input_data)

# =====================================================
# Prepare Decoder Target Data
# =====================================================

decoder_target_data = np.zeros_like(decoder_input_data)

decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

decoder_target_data = np.expand_dims(
    decoder_target_data,
    axis=-1
)

print("\nDecoder Target Data Shape:")
print(decoder_target_data.shape)

# =====================================================
# Build Encoder
# =====================================================

tf.keras.backend.clear_session()

latent_dim = 64

encoder_inputs = Input(shape=(None,))

encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=latent_dim
)(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)(encoder_embedding)

# =====================================================
# Build Decoder
# =====================================================

decoder_inputs = Input(shape=(None,))

decoder_embedding = Embedding(
    input_dim=fra_vocab_size,
    output_dim=latent_dim
)(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

# =====================================================
# Output Layer
# =====================================================

decoder_dense = Dense(
    fra_vocab_size,
    activation="softmax"
)

outputs = decoder_dense(decoder_outputs)

# =====================================================
# Create Model
# =====================================================

model = Model(
    [encoder_inputs, decoder_inputs],
    outputs
)

# =====================================================
# Compile Model
# =====================================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nModel Summary:")
model.summary()

# =====================================================
# Train Model
# =====================================================

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=50,
    verbose=1
)

print("\nTraining Completed Successfully!")

English Vocabulary Size: 11
French Vocabulary Size: 12

English Vocabulary:
{'i': 1, 'love': 2, 'ai': 3, 'deep': 4, 'learning': 5, 'how': 6, 'are': 7, 'you': 8, 'good': 9, 'morning': 10}

French Vocabulary:
{'start': 1, 'end': 2, 'j': 3, 'aime': 4, 'ai': 5, 'apprentissage': 6, 'profond': 7, 'comment': 8, 'allez': 9, 'vous': 10, 'bonjour': 11}

Encoder Sequences:
[[1, 2, 3], [1, 2, 4, 5], [6, 7, 8], [9, 10]]

Decoder Sequences:
[[1, 3, 4, 5, 2], [1, 3, 4, 6, 7, 2], [1, 8, 9, 10, 2], [1, 11, 2]]

Encoder Input Data:
[[ 1  2  3  0]
 [ 1  2  4  5]
 [ 6  7  8  0]
 [ 9 10  0  0]]

Decoder Input Data:
[[ 1  3  4  5  2  0]
 [ 1  3  4  6  7  2]
 [ 1  8  9 10  2  0]
 [ 1 11  2  0  0  0]]

Decoder Target Data Shape:
(4, 6, 1)

Model Summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │        704 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │        768 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │     33,024 │ embedding[0][0]   │
│                     │ 64), (None, 64),  │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     33,024 │ embedding_1[0][0… │
│                     │ 64), (None, 64),  │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 12)  │        780 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 68,300 (266.80 KB)

 Trainable params: 68,300 (266.80 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.1250 - loss: 2.4824
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5833 - loss: 2.4658
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5000 - loss: 2.4519
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.4167 - loss: 2.4345
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.3750 - loss: 2.4143
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3750 - loss: 2.3897
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3750 - loss: 2.3668
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3750 - loss: 2.3349
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 2.2893
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3750 - loss: 2.2470
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 2.1743
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 2.1067
E

In [46]:
reverse_eng_index = {
    v: k for k, v in eng_tokenizer.word_index.items()
}

reverse_fra_index = {
    v: k for k, v in fra_tokenizer.word_index.items()
}

In [47]:
def translate_sentence(sentence):

    # Convert input sentence to sequence
    seq = eng_tokenizer.texts_to_sequences([sentence])

    # Pad to encoder length
    seq = pad_sequences(
        seq,
        maxlen=encoder_input_data.shape[1],
        padding='post'
    )

    # Start token
    start_token = fra_tokenizer.word_index['start']

    decoder_input = np.array([[start_token]])

    translated_words = []

    for _ in range(10):  # maximum output length

        prediction = model.predict(
            [seq, decoder_input],
            verbose=0
        )

        predicted_id = np.argmax(
            prediction[0, -1, :]
        )

        predicted_word = reverse_fra_index.get(
            predicted_id,
            ''
        )

        if predicted_word == 'end':
            break

        translated_words.append(predicted_word)

        decoder_input = np.concatenate(
            [decoder_input, [[predicted_id]]],
            axis=1
        )

    return " ".join(translated_words)

In [48]:
def translate_sentence(sentence):

    # Convert input sentence to sequence
    seq = eng_tokenizer.texts_to_sequences([sentence])

    # Pad to encoder length
    seq = pad_sequences(
        seq,
        maxlen=encoder_input_data.shape[1],
        padding='post'
    )

    # Start token
    start_token = fra_tokenizer.word_index['start']

    decoder_input = np.array([[start_token]])

    translated_words = []

    for _ in range(10):  # maximum output length

        prediction = model.predict(
            [seq, decoder_input],
            verbose=0
        )

        predicted_id = np.argmax(
            prediction[0, -1, :]
        )

        predicted_word = reverse_fra_index.get(
            predicted_id,
            ''
        )

        if predicted_word == 'end':
            break

        translated_words.append(predicted_word)

        decoder_input = np.concatenate(
            [decoder_input, [[predicted_id]]],
            axis=1
        )

    return " ".join(translated_words)

In [51]:
print(translate_sentence("i love ai"))

print(translate_sentence("i love deep learning"))

print(translate_sentence("how are you"))

print(translate_sentence("good morning"))

print(translate_sentence("who are you"))

j aime
j aime aime profond
comment
bonjour
comment
